# Food lists creation

This notebook processes the [FNDDS foods list](https://www.ars.usda.gov/northeast-area/beltsville-md-bhnrc/beltsville-human-nutrition-research-center/food-surveys-research-group/docs/fndds-download-databases/) and create final lists of foods across different categories, to be used in the benchmark construction task (task 2). It utilizes the FNDDS food category codes which can be found in page 39-43 of [this document](https://www.ars.usda.gov/ARSUserFiles/80400530/pdf/fndds/2021_2023_FNDDS_Doc.pdf). 

The data files needed to run this notebook should be placed in a `data` folder under the same directory, which contains these files: 

* `food_list.csv`: a list of 9640 foods found in the FNDDS section of the USDA website. 
* `food_tagging.csv`: detailed nutritional information about these 9640 food items. 

In the same directory, there should also be a `processed_data` folder to store the output files corresponding to the below **7 main categories** of foods: 

* `reduced_mixed_dishes.csv`: 662 food items
* `reduced_meat_seafood.csv`: 192 food items
* `reduced_processed_meat.csv`: 67 food items
* `reduced_plant_protein.csv`: 33 food items
* `reduced_breads.csv`: 152 food items
* `reduced_baked_desserts.csv`: 189 food items
* `reduced_vegetables_potatoes.csv`: 151 food items

Details on the filtering process can be found in each section below. 

## Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import random
import os
import time
from autogen import ConversableAgent
from dotenv import load_dotenv
import logging
import requests
from bs4 import BeautifulSoup
import csv
from utils import concat_data_across_years
import warnings
warnings.filterwarnings('ignore')

## Step 0: Create a list of foods, with nutrition tags

Source: FNDDS and NHANES

For this step, we aim to create a list of foods to be used in downstream tasks, with information such as name of the food, food category, ingredients, nutritional tags. While FDNDDS provides us with the list of foods, it lacks the nutritional info, which we then get from NHANES. 

### FNDDS data (2015-2016, 2017-2018, 2019-2020)

In [ ]:
# The most recent three years of FNDDS tables contain nutrition data. The column names are slightly different.
df_1516 = pd.read_excel('../data/2015-2016 Ingredients.xlsx', skiprows=1)
df_1718 = pd.read_excel('../data/2017-2018 Ingredients.xlsx', skiprows=1)
df_1920 = pd.read_excel('../data/2019-2020 Ingredients.xlsx', skiprows=1)

# Unify the column names.
df_1516 = df_1516.rename(columns={'WWEIA Category code': 'WWEIA Category number'})

# A small proportion of FNDDS data, such as code and descriptions change over the years.
# Here we take the latest version of data if there are duplicates.
df_fndds = pd.concat([df_1516, df_1718, df_1920])
df_fndds = df_fndds[['Food code', 'Main food description', 'WWEIA Category number', 'WWEIA Category description', 'Ingredient code', 'Ingredient description']]
df_fndds = df_fndds.drop_duplicates(subset=['Food code', 'WWEIA Category number', 'Ingredient code'], keep='last')
df_fndds = df_fndds.sort_values(by='Food code')

# This table records the connections between food and ingredients.
df_fndds = df_fndds.rename(columns={'Food code': 'food_id', 'Main food description': 'food_desc', 'WWEIA Category number': 'WWEIA_id',
                        'WWEIA Category description': 'WWEIA_desc', 'Ingredient code': 'ingredient_id', 'Ingredient description': 'ingredient_desc'})

In [ ]:
# There are 9260 foods in total.
# This is the FNDDS dataset. 
print(len(set(df_fndds['food_id'].tolist())))
df_fndds.head()

In [ ]:
df_fndds.to_csv('../processed_data/fndds.csv', index=False)

### Dietary Record Data (NHANES)

Periods: 2003-2004, 2005-2006, 2007-2008, 2009-2010, 2011-2012, 2013-2014, 2015-2016, 2017-2018, 2017-2020

In [ ]:
years = ['0304', '0506', '0708', '0910', '1112', '1314', '1516', '1718', '1720']
year_char = 'C'
type_dietary = 'dietary'

df_IFF1 = concat_data_across_years(type_dietary, 'DR1IFF', years, year_char)
df_IFF2 = concat_data_across_years(type_dietary, 'DR2IFF', years, year_char)

# Food and nutrition data
food_columns_1 = ['SEQN', 'food_id', 'DR1IGRMS',
 'DR1IKCAL', 'DR1IPROT', 'DR1ICARB', 'DR1ISUGR', 'DR1IFIBE', 'DR1ITFAT',
 'DR1ISFAT', 'DR1IMFAT', 'DR1IPFAT', 'DR1ICHOL', 'DR1IATOC', 'DR1IATOA',
 'DR1IRET', 'DR1IVARA', 'DR1IACAR', 'DR1IBCAR', 'DR1ICRYP', 'DR1ILYCO',
 'DR1ILZ', 'DR1IVB1', 'DR1IVB2', 'DR1INIAC', 'DR1IVB6', 'DR1IFOLA',
 'DR1IFA', 'DR1IFF', 'DR1IFDFE', 'DR1ICHL', 'DR1IVB12', 'DR1IB12A',
 'DR1IVC', 'DR1IVD', 'DR1IVK', 'DR1ICALC', 'DR1IPHOS', 'DR1IMAGN',
 'DR1IIRON', 'DR1IZINC', 'DR1ICOPP', 'DR1ISODI', 'DR1IPOTA', 'DR1ISELE',
 'DR1ICAFF', 'DR1ITHEO', 'DR1IALCO', 'DR1IMOIS'
]
food_columns_2 = ['SEQN', 'food_id', 'DR2IGRMS',
 'DR2IKCAL', 'DR2IPROT', 'DR2ICARB', 'DR2ISUGR', 'DR2IFIBE', 'DR2ITFAT',
 'DR2ISFAT', 'DR2IMFAT', 'DR2IPFAT', 'DR2ICHOL', 'DR2IATOC', 'DR2IATOA',
 'DR2IRET', 'DR2IVARA', 'DR2IACAR', 'DR2IBCAR', 'DR2ICRYP', 'DR2ILYCO',
 'DR2ILZ', 'DR2IVB1', 'DR2IVB2', 'DR2INIAC', 'DR2IVB6', 'DR2IFOLA',
 'DR2IFA', 'DR2IFF', 'DR2IFDFE', 'DR2ICHL', 'DR2IVB12', 'DR2IB12A',
 'DR2IVC', 'DR2IVD', 'DR2IVK', 'DR2ICALC', 'DR2IPHOS', 'DR2IMAGN',
 'DR2IIRON', 'DR2IZINC', 'DR2ICOPP', 'DR2ISODI', 'DR2IPOTA', 'DR2ISELE',
 'DR2ICAFF', 'DR2ITHEO', 'DR2IALCO', 'DR2IMOIS'
]

df_IFF1 = df_IFF1.rename(columns={'DR1IFDCD': 'food_id'})
df_IFF1 = df_IFF1[food_columns_1].astype(float)
df_IFF2 = df_IFF2.rename(columns={'DR2IFDCD': 'food_id'})
df_IFF2 = df_IFF2[food_columns_2].astype(float)
df_food  = pd.DataFrame(np.vstack((df_IFF1.to_numpy(), df_IFF2.to_numpy())), columns=df_IFF1.columns)

df_IFF1 = df_IFF1[['SEQN', 'food_id']].astype(int).astype(str)
df_IFF2 = df_IFF2[['SEQN', 'food_id']].astype(int).astype(str)
df_IFF1['food_id'] = df_IFF1['food_id'].str.zfill(10)
df_IFF2['food_id'] = df_IFF2['food_id'].str.zfill(10)
df_food_user = pd.concat([df_IFF1, df_IFF2])

In [ ]:
# This is the crosswalk between users and food records.
df_food_user

In [ ]:
df_food_user.to_csv('../processed_data/food_user.csv', index=False)

In [ ]:
# Create a new DataFrame for the nutritional data
df_nutrition = pd.DataFrame()
df_nutrition['food_id'] = df_food['food_id'].unique()
df_food = df_food.dropna(subset=['DR1IGRMS'])
for col in df_food.columns.tolist()[3:]:
    df_food[col] = df_food[col] / df_food['DR1IGRMS'] * 100

df_food.drop(['SEQN', 'DR1IGRMS'], axis=1, inplace=True)

df_food = df_food.groupby('food_id').mean().reset_index()
df_food = df_food.fillna(0)
df_food['food_id'] = df_food['food_id'].astype(int)
df_nutrition = df_nutrition.merge(df_food, how='left', on='food_id')

In [ ]:
df_nutrition.head()

In [ ]:
"""
This is merely for information. Not used in the pipeline.

We use the food code NHANES provided, which is more complete than FNDDS. For duplications, we also keep the latest records.
In this way, every food users reported has its corresponding food description.
We use this as the connections between users and food.
"""

food_dictionary = concat_data_across_years(type_dietary, 'DRXFCD', years, year_char)
food_dictionary = food_dictionary.rename(columns={'DRXFDCD': 'food_id', 'DRXFCLD': 'food_desc'})

food_dictionary = food_dictionary[['food_id', 'food_desc', 'years']]
food_dictionary['food_id'] = food_dictionary['food_id'].astype(int)
food_dictionary = food_dictionary.drop_duplicates(subset='food_id', keep='last')

food_nhanes_have = set(food_dictionary['food_id'].tolist())
len(food_nhanes_have)

By now, we have three tables: 

* `df_fndds` is the table for connecting foods to ingredients and categories; 

*  `df_food_user` is the table for connecting foods to users who consume them; 
 
*  `df_nutrition` is the table for the foods and their nutritions per 100g.

In [ ]:
df_nutrition['food_id'] = df_nutrition['food_id'].astype(int).astype(str).str.zfill(10)
df_nutrition = df_nutrition.set_index('food_id')
df_nutrtion = df_nutrition.round(2)

In [ ]:
df_nutrition.describe().round(2)

### Tagging the food items

In [ ]:
nutrition_mapping = {'DR1IKCAL': 'calorie', 'DR1IPROT': 'protein', 'DR1ICARB': 'carb', 'DR1ISUGR': 'sugar', 'DR1IFIBE': 'fiber', 
                     'DR1ISFAT': 'saturated_fat', 'DR1ICHOL': 'cholesterol', 'DR1ISODI': 'sodium', 'DR1ICALC': 'calcium', 'DR1IPHOS': 'phosphorus',
                     'DR1IPOTA': 'potassium', 'DR1IIRON': 'iron', 'DR1IFA': 'folic_acid', 'DR1IVC': 'vitamin_c', 'DR1IVD': 'vitamin_d', 'DR1IVB12': 'vitamin_b12'
                     }
nutrition_columns = ['DR1IKCAL', 'DR1IPROT', 'DR1ICARB', 'DR1ISUGR', 'DR1IFIBE', 'DR1ISFAT', 'DR1ICHOL', 
                    'DR1ISODI', 'DR1ICALC', 'DR1IPHOS', 'DR1IPOTA', 'DR1IIRON', 'DR1IFA', 'DR1IVC', 'DR1IVD', 'DR1IVB12']

In [ ]:
thresholds = {
    'calorie': {'low': 40, 'high': 225},
    'protein': {'low': 10, 'high': 15},
    'carb': {'low': 55, 'high': 75},
    'sugar': {'low': 5, 'high': 22.5},
    'fiber': {'low': 3, 'high': 6},
    'saturated_fat': {'low': 1.5, 'high': 5},
    'cholesterol': {'low': 20, 'high': 40},
    'sodium': {'low': 120, 'high': 200},
    'calcium': {'low': 0, 'high': 150},
    'phosphorus': {'low': 0, 'high': 105},
    'potassium': {'low': 0, 'high': 525},
    'iron': {'low': 0, 'high': 3.3},
    'folic_acid': {'low': 0, 'high': 60},
    'vitamin_c': {'low': 0, 'high': 15},
    'vitamin_d': {'low': 0, 'high': 2.25},
    'vitamin_b12': {'low': 0, 'high': 0.36},
}

In [ ]:
df_nutrition = df_nutrition[nutrition_columns]
df_nutrition = df_nutrition.rename(columns=nutrition_mapping)
for nutrient, cols in nutrition_mapping.items():
    low_col = f'low_{cols}'
    high_col = f'high_{cols}'
    
    df_nutrition[low_col] = df_nutrition[cols].apply(lambda x: 1 if x <= thresholds[cols]['low'] else 0)
    df_nutrition[high_col] = df_nutrition[cols].apply(lambda x: 1 if x > thresholds[cols]['high'] else 0)

In [ ]:
df_nutrition.describe().T

In [ ]:
df_nutrition.to_csv('../processed_data/food_tagging.csv')

### Additional FNDDS food codes from 2007-2008

Since `fndds.csv` only contains FNDDS data from 2015 onwards, but `food_tagging.csv` contains NHANES data from 2003 onwards, when we attempt to left join `food_tagging` with `fndds`, there are a lot of missing values (~1.5k food codes in `food_tagging` don't have a match in `fndds`).

While researching into old FNDDS food codes pre-2015, we came across this source that thas FNDDS food codes in 2007-2008. Upon joining `food_tagging` on both `fndds` and this source, the number of food codes with missing data is down to 291. For these food codes, we drop entirely from the foods list since early-day FNDDS data is very limited. This removal is trivial since we're still left with ~9.3k food items. 

In this section, we first scrape from the table here: https://www.cdc.gov/mmwr/preview/mmwrhtml/mm6105-table.htm to get FNDDS food codes from 2007-2008 period

In [ ]:
url = "https://www.cdc.gov/mmwr/preview/mmwrhtml/mm6105-table.htm"

response = requests.get(url)

if response.status_code == 200:

    soup = BeautifulSoup(response.content, "html.parser")    

    # Extract all tables, get first table
    tables = soup.find_all("table")
    target_table = tables[0]
    
    # Extract rows from the table
    rows = target_table.find_all("tr")
    
    table_data = []
    for row in rows:
        
        # Extract all cells (th or td)
        cells = row.find_all(["th", "td"])
        
        # Get the text from each cell
        row_data = [cell.get_text(strip=True) for cell in cells]
        table_data.append(row_data)
    
    df = pd.DataFrame(table_data)

    # Column headers    
    df.columns = df.iloc[0]
    df = df[1:]  # Drop the header row from the data

    df.rename(
        columns={
            "Group no.": "group_id",
            "Category no.": "WWEIA_id",
            "Group": "group_desc",
            "Food category": "WWEIA_desc",
            "FNDDS code": "food_id",
            "Food description": "food_desc",
        },
        inplace=True,
    )
    
    output_file = "../processed_data/fndds_food_codes_0708.csv"
    df.to_csv(output_file, index=False)
    print(f"Success!")
else:
    print(f"Failed to fetch webpage. Status code: {response.status_code}")


In [ ]:
fndds_food_codes_file = "../processed_data/fndds_food_codes_0708.csv"
fndds_food_codes = pd.read_csv(fndds_food_codes_file)

# Extract unique combinations of WWEIA_id and WWEIA_desc
unique_wweia_codes = fndds_food_codes[["WWEIA_id", "WWEIA_desc", "group_id", "group_desc"]].drop_duplicates()
unique_wweia_codes = unique_wweia_codes.dropna(subset=["WWEIA_id", "WWEIA_desc", "group_id", "group_desc"])

output_file = "../processed_data/WWEIA_codes_0708.csv"
unique_wweia_codes.to_csv(output_file, index=False)

### Joining `food_tagging` and `fndds`

In [ ]:
food_tagging = pd.read_csv('../processed_data/food_tagging.csv')
fndds = pd.read_csv('../processed_data/fndds.csv')

# Ensure the food_id column in both dataframes is of the same type (string)
food_tagging['food_id'] = food_tagging['food_id'].astype(str).str.lstrip('00')
fndds['food_id'] = fndds['food_id'].astype(str)

# Remove duplicates in fndds by keeping the first occurrence of each food_id
fndds = fndds.drop_duplicates(subset='food_id', keep='first')

foods_list = food_tagging.merge(fndds, on='food_id', how='left')

foods_list = foods_list[['food_id', 'food_desc', 'WWEIA_id', 'WWEIA_desc', 'ingredient_id', 'ingredient_desc']]

foods_list.to_csv('../processed_data/initial_foods_list.csv', index=False)

In [ ]:
foods_list = pd.read_csv('../processed_data/initial_foods_list.csv')

# Count rows where food_desc is empty (NaN or empty string)
empty_food_desc_count = foods_list['food_desc'].isna().sum() + (foods_list['food_desc'] == '').sum()

print(f"Number of rows with empty food_desc: {empty_food_desc_count}")

### Joining with `fndds_food_codes_0708` to get final `foods_list`

In [ ]:
foods_list_file = "../processed_data/initial_foods_list.csv"
fndds_food_codes_file = "../processed_data/fndds_food_codes_0708.csv"

foods_list = pd.read_csv(foods_list_file)
fndds_food_codes = pd.read_csv(fndds_food_codes_file)

fndds_food_codes = fndds_food_codes[["food_id", "food_desc", "WWEIA_id", "WWEIA_desc"]]

merged_df = pd.merge(
    foods_list,
    fndds_food_codes,
    on="food_id",
    how="left",
    suffixes=("", "_fndds")  # To avoid column name clashes
)

merged_df["food_desc"] = merged_df["food_desc"].fillna(merged_df["food_desc_fndds"])
merged_df["WWEIA_id"] = merged_df["WWEIA_id"].fillna(merged_df["WWEIA_id_fndds"])
merged_df["WWEIA_desc"] = merged_df["WWEIA_desc"].fillna(merged_df["WWEIA_desc_fndds"])

merged_df.drop(columns=["food_desc_fndds", "WWEIA_id_fndds", "WWEIA_desc_fndds"], inplace=True)

output_file = "../processed_data/foods_list.csv"
merged_df.to_csv(output_file, index=False)

In [ ]:
foods_list = pd.read_csv('../processed_data/foods_list.csv')

# Count rows where food_desc is empty (NaN or empty string)
empty_food_desc_count = foods_list['food_desc'].isna().sum() + (foods_list['food_desc'] == '').sum()

print(f"Number of rows with empty food_desc: {empty_food_desc_count}")

In [ ]:
foods_list_file = "../processed_data/foods_list.csv"

foods_list = pd.read_csv(foods_list_file)

# Remove rows where 'food_desc' is empty (NaN or blank)
foods_list = foods_list.dropna(subset=["food_desc"])

# Save the cleaned dataset back to the same file
foods_list.to_csv(foods_list_file, index=False)

In [ ]:
foods_list = pd.read_csv('../processed_data/foods_list.csv')

# Count rows where food_desc is empty (NaN or empty string)
empty_food_desc_count = foods_list['food_desc'].isna().sum() + (foods_list['food_desc'] == '').sum()

print(f"Number of rows with empty food_desc: {empty_food_desc_count}")

## Filtering step 1: Applicable to all 7 food categories:

* We first filter for items from sub-categories (`WWEIA_desc`) under a specific food category. 
    
    - For example, mixed dishes can be identified through these WWEIA descriptions, to name a few: 'Meat mixed dishes', 'Poultry mixed dishes', 'Seafood mixed dishes', 'Bean, pea, legume dishes', 'Vegetable dishes', etc. 

* We then remove special characters from the food description and get the first `n` number of foods (usually 2 or 3 depending on the category and on how intense we want the filtering to be, with higher `num_first_words` corresponding to less intense filtering). The purpose of this step is to make sure we can remove near-similar foods that might not necessarily have the exact same name, but they can be considered duplicates. Foods with the same first few words will be grouped together and undergo a selection process. 

    - For example, these foods are almost the same, with the only difference being the type of protein: 58136140 "Lo mein, with pork", 58136130 "Lo mein, with shrimp", 58136150 "Lo mein, with beef", 58136160 "Lo mein, with chicken". To make sure the we have a manageable foods list for latter tasks, we can choose to keep only one food item out of this list of near-similar foods. 

* After grouping foods with similar first few words together, we will select which food to keep (usually one food item remains for most cases, but there might be more than one food item being kept if there is a tie). The general idea is we want to keep foods that have the most nutritional information, that is, the most number of nutrition tags, since they will be helpful in constructing question-answer pairs of varying difficulty levels. The criteria are as follows: 

    - We first look at 16 primary nutrition tags (8 main categories: calorie, protein, carb, sugar, fiber, saturated fat, cholesterol, sodium - each with 2 columns corresponding to high vs. low levels) and get the foods with the highest count in these 16 primary nutrition columns.

    - If there is a tie, we then consider 16 secondary nutrition tags (8 main categories: calcium, phosphorous, potassium, iron, folic acid, vitamin C, vitamin D, vitamin B12 - each with 2 columns corresponding to high vs. low levels) and get the foods with the highest count in these 16 secondary nutrition columns. 

    - If there is still a tie, we randomly select one food item. 

* For many food categories (specifically: mixed dishes, processed meat, baked desserts, breads), no more filtering is needed after this step, since all food items selected at the end of this step can be used in the recommender system already. 

* However, for other categories (meat & seafood, plant protein, vegetables & potatoes), we need to execute another filtering step to make sure we remove foods that cannot be used in a recommender system, such as raw foods, or foods that have not been sufficiently processed or cooked and not yet ready for human consumption.

In [ ]:
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

def process_foods(category, wweia_desc_filters, num_first_words):
    file_path = '../processed_data/foods_list.csv'
    foods_df = pd.read_csv(file_path)
    
    # Filter foods_df for rows where WWEIA_desc is in the specified list
    filtered_foods_df = foods_df[foods_df['WWEIA_desc'].isin(wweia_desc_filters)]
    
    output_file_path = f'../processed_data/{category}.csv'
    filtered_foods_df.to_csv(output_file_path, index=False)
    
    print("Original food list before processing:")
    print(f"Number of rows: {len(filtered_foods_df)}")
    print(f"Number of unique food descriptions: {filtered_foods_df['food_desc'].nunique()}")
    print(f"Number of unique food id's: {filtered_foods_df['food_id'].nunique()}")

    mixed_dishes_df = pd.read_csv(output_file_path)
    food_tagging_df = pd.read_csv('../processed_data/food_tagging.csv')

    # Remove special characters and get the first few words of a food description
    def get_first_words(food_desc, num_words=3):
        clean_desc = re.sub(r'[^a-zA-Z0-9\s]', '', food_desc).lower()
        return ' '.join(clean_desc.split()[:num_words])

    mixed_dishes_df['first_words'] = mixed_dishes_df['food_desc'].apply(lambda x: get_first_words(x, num_words=num_first_words))

    # Group foods by their first few words
    grouped_foods = mixed_dishes_df.groupby('first_words')

    # Count the number of non-zero nutritional tags in specified columns
    def count_non_zero_nutrition_tags(row, columns):
        return row[columns].sum()

    # Primary and secondary sets of nutritional columns
    secondary_nutrition_columns = [
        'low_calcium', 'high_calcium',
        'low_phosphorus', 'high_phosphorus', 'low_potassium', 'high_potassium', 'low_iron', 'high_iron',
        'low_folic_acid', 'high_folic_acid', 'low_vitamin_c', 'high_vitamin_c', 'low_vitamin_d', 'high_vitamin_d',
        'low_vitamin_b12', 'high_vitamin_b12'
    ]

    primary_nutrition_columns = [
        'low_calorie', 'high_calorie', 'low_protein', 'high_protein', 'low_carb', 'high_carb',
        'low_sugar', 'high_sugar', 'low_fiber', 'high_fiber', 'low_saturated_fat', 'high_saturated_fat',
        'low_cholesterol', 'high_cholesterol', 'low_sodium', 'high_sodium'
    ]

    # Pick one food from each group based on the nutritional tag count
    selected_foods = []

    for group_name, group in grouped_foods:
        group_food_ids = group['food_id']
        
        group_foods_with_tags = pd.merge(
            group, 
            food_tagging_df, 
            on='food_id',
            how='inner'
        )
        
        # Find the food with the most non-zero nutritional tags (primary set)
        group_foods_with_tags['non_zero_primary_count'] = group_foods_with_tags.apply(count_non_zero_nutrition_tags, axis=1, columns=primary_nutrition_columns)
        
        # Get the foods with the highest count in primary nutrition columns
        top_primary_foods = group_foods_with_tags.sort_values(by='non_zero_primary_count', ascending=False)
        top_primary_count = top_primary_foods['non_zero_primary_count'].iloc[0]
        tied_foods = top_primary_foods[top_primary_foods['non_zero_primary_count'] == top_primary_count]
        
        # If there's a tie, compare by the secondary set of nutritional columns
        if len(tied_foods) > 1:
            tied_foods['non_zero_secondary_count'] = tied_foods.apply(count_non_zero_nutrition_tags, axis=1, columns=secondary_nutrition_columns)
            top_secondary_foods = tied_foods.sort_values(by='non_zero_secondary_count', ascending=False)
            top_secondary_count = top_secondary_foods['non_zero_secondary_count'].iloc[0]
            tied_foods_secondary = top_secondary_foods[top_secondary_foods['non_zero_secondary_count'] == top_secondary_count]
            
            # If there's still a tie, randomly choose one
            if len(tied_foods_secondary) > 1:
                selected_food = tied_foods_secondary.sample(1).iloc[0]
            else:
                selected_food = tied_foods_secondary.iloc[0]
        else:
            selected_food = tied_foods.iloc[0]

        selected_foods.append(selected_food)

    selected_foods_df = pd.DataFrame(selected_foods)

    selected_foods_df = selected_foods_df.drop_duplicates(subset=['food_id'])

    selected_foods_df = selected_foods_df.drop_duplicates(subset=['food_desc'])

    # Columns to keep in the final file
    columns_to_keep = [
        "food_id", "food_desc", "WWEIA_desc", "ingredient_desc", "calorie", "protein", "carb", 
        "sugar", "fiber", "saturated_fat", "cholesterol", "sodium", "calcium", "phosphorus", 
        "potassium", "iron", "folic_acid", "vitamin_c", "vitamin_d", "vitamin_b12", 
        "low_calorie", "high_calorie", "low_protein", "high_protein", "low_carb", "high_carb", 
        "low_sugar", "high_sugar", "low_fiber", "high_fiber", "low_saturated_fat", 
        "high_saturated_fat", "low_cholesterol", "high_cholesterol", "low_sodium", "high_sodium", 
        "low_calcium", "high_calcium", "low_phosphorus", "high_phosphorus", "low_potassium", 
        "high_potassium", "low_iron", "high_iron", "low_folic_acid", "high_folic_acid", 
        "low_vitamin_c", "high_vitamin_c", "low_vitamin_d", "high_vitamin_d", 
        "low_vitamin_b12", "high_vitamin_b12"
    ]

    selected_foods_df = selected_foods_df[columns_to_keep]

    print("------------------------------\nFood list after processing: ")
    print(f"Number of rows: {len(selected_foods_df)}")
    print(f"Number of unique food descriptions: {selected_foods_df['food_desc'].nunique()}")
    print(f"Number of unique food id's: {selected_foods_df['food_id'].nunique()}")


    selected_foods_df.to_csv(f'../processed_data/reduced_{category}.csv', index=False)
    
    print(f"------------------------------\nProcessed category '{category}' and saved reduced food list.")


### 1. Mixed dishes (final, no more filtering needed)

In [ ]:
# Load the WWEIA_codes_0708 dataset
wweia_codes_file = "../processed_data/WWEIA_codes_0708.csv"
wweia_codes = pd.read_csv(wweia_codes_file)

# Filter rows where category_desc is "MIXED DISHES"
mixed_dishes = wweia_codes[wweia_codes["group_desc"] == "MIXED DISHES"]

# Extract the WWEIA_desc column as a list
mixed_dishes_list = mixed_dishes["WWEIA_desc"].tolist()

print("List of WWEIA_desc for MIXED DISHES:")
print(mixed_dishes_list)

In [ ]:
category = "mixed_dishes"
wweia_desc_filters = [
    'Meat mixed dishes',
    'Poultry mixed dishes',
    'Seafood mixed dishes',
    'Bean, pea, legume dishes',
    'Vegetable dishes',
    'Rice mixed dishes',
    'Pasta mixed dishes, excludes macaroni & cheese',
    'Macaroni and cheese',
    'Turnovers and other grain-based items',
    'Fried rice and lo/chow mein',
    'Stir-fry and soy-based sauce mixtures',
    'Egg rolls, dumplings, sushi',
    'Burritos and tacos',
    'Nachos',
    'Other Mexican mixed dishes',
    'Pizza',
    'Burgers',
    'Frankfurter sandwiches',
    'Chicken fillet sandwiches',
    'Egg/breakfast sandwiches',
    'Cheese sandwiches',
    'Peanut butter and jelly sandwiches',
    'Seafood sandwiches',
    'Deli and cured meat sandwiches',
    'Meat and BBQ sandwiches',
    'Vegetable sandwiches/burgers',
    'Soups, broth-based',
    'Soups, cream-based',
    'Ramen and Asian broth-based soups',

    # 0708 codes
    'Meat mixed dishes',
    'Poultry mixed dishes',
    'Fish and seafood mixed dishes',
    'Sandwiches (single code)',
    'Egg rolls and filled dough items',
    'Burritos, tacos, tamales',
    'Fried rice, lo mein, stir-fry mixtures',
    'Pizza',
    'Soups',
    'Pasta mixed dishes, excludes macaroni and cheese',
    'Macaroni and cheese',
    'Rice mixed dishes'
]

process_foods(category, wweia_desc_filters, num_first_words=2)

### 2. Meat & seafood (not final, additional filtering needed)

In [ ]:
# Load the WWEIA_codes_0708 dataset
wweia_codes_file = "../processed_data/WWEIA_codes_0708.csv"
wweia_codes = pd.read_csv(wweia_codes_file)

# Filter rows where category_desc is "MIXED DISHES"
mixed_dishes = wweia_codes[wweia_codes["group_desc"] == "PROTEIN FOODS"]

# Extract the WWEIA_desc column as a list
mixed_dishes_list = mixed_dishes["WWEIA_desc"].tolist()

print("List of WWEIA_desc for PROTEIN FOODS:")
display(mixed_dishes_list)

In [ ]:
category = "meat_seafood"
wweia_desc_filters = [
'Beef, excludes ground',
'Ground beef',
'Pork',
'Lamb, goat, game',
'Liver and organ meats',
'Chicken, whole pieces',
'Chicken patties, nuggets and tenders',
'Turkey, duck, other poultry',
'Fish',
'Shellfish',
'Eggs and omelets',

# extra codes from 0708
'Beef, excludes ground',
'Ground beef',
'Pork',
'Other meats',
'Poultry',
'Fish and seafood',
'Eggs and egg mixed dishes'
]

process_foods(category, wweia_desc_filters, num_first_words=2)

### 3. Processed meat (final, no more filtering needed)

In [ ]:
# Load the WWEIA_codes_0708 dataset
wweia_codes_file = "../processed_data/WWEIA_codes_0708.csv"
wweia_codes = pd.read_csv(wweia_codes_file)

# Filter rows where category_desc is "MIXED DISHES"
mixed_dishes = wweia_codes[wweia_codes["group_desc"] == "PROTEIN FOODS"]

# Extract the WWEIA_desc column as a list
mixed_dishes_list = mixed_dishes["WWEIA_desc"].tolist()

print("List of WWEIA_desc for PROTEIN FOODS:")
display(mixed_dishes_list)

In [ ]:
category = "processed_meat"
wweia_desc_filters = [
'Cold cuts and cured meats',
'Bacon',
'Frankfurters',
'Sausages',

# extra codes from 0708
'Cold cuts and cured meats',
'Frankfurters and sausages',
'Bacon'
]

process_foods(category, wweia_desc_filters, num_first_words=2)

### 4. Plant protein (not final, additional filtering needed)

In [ ]:
# Load the WWEIA_codes_0708 dataset
wweia_codes_file = "../processed_data/WWEIA_codes_0708.csv"
wweia_codes = pd.read_csv(wweia_codes_file)

# Filter rows where category_desc is "MIXED DISHES"
mixed_dishes = wweia_codes[wweia_codes["group_desc"] == "PROTEIN FOODS"]

# Extract the WWEIA_desc column as a list
mixed_dishes_list = mixed_dishes["WWEIA_desc"].tolist()

print("List of WWEIA_desc for PROTEIN FOODS:")
display(mixed_dishes_list)

In [ ]:
category = "plant_protein"
wweia_desc_filters = [
'Beans, peas, legumes',
'Nuts and seeds',
'Soy and meat-alternative products',

# extra codes from 0708
'Processed soy products',
'Legumes and legume mixed dishes',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

### 1.5. Breads & Grains (final, no additional filtering needed)

In [ ]:
# Load the WWEIA_codes_0708 dataset
wweia_codes_file = "../processed_data/WWEIA_codes_0708.csv"
wweia_codes = pd.read_csv(wweia_codes_file)

# Filter rows where category_desc is "MIXED DISHES"
mixed_dishes = wweia_codes[wweia_codes["group_desc"] == "GRAINS"]

# Extract the WWEIA_desc column as a list
mixed_dishes_list = mixed_dishes["WWEIA_desc"].tolist()

print("List of WWEIA_desc for GRAINS:")
display(mixed_dishes_list)

In [ ]:
category = "breads_grains"
wweia_desc_filters = [
'Yeast breads',
'Rolls and buns',
'Bagels and English muffins',
'Tortillas',
'Biscuits, muffins, quick breads',
'Pancakes, waffles, French toast',

# extra codes from 0708
'Pasta, noodles, other grains',
'Breads and rolls',
'Tortillas',
'Biscuits, muffins, quick breads',
'Pancakes, waffles, French toast',
'Ready-to-eat cereals',
'Cooked cereals',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

### 1.6. Baked Desserts (final, no additional filtering needed)

In [ ]:
# Load the WWEIA_codes_0708 dataset
wweia_codes_file = "../processed_data/WWEIA_codes_0708.csv"
wweia_codes = pd.read_csv(wweia_codes_file)

# Filter rows where category_desc is "MIXED DISHES"
mixed_dishes = wweia_codes[wweia_codes["group_desc"] == "GRAINS"]

# Extract the WWEIA_desc column as a list
mixed_dishes_list = mixed_dishes["WWEIA_desc"].tolist()

print("List of WWEIA_desc for GRAINS:")
display(mixed_dishes_list)

In [ ]:
category = "baked_desserts"
wweia_desc_filters = [
'Cakes and pies',
'Cookies and brownies',
'Doughnuts, sweet rolls, pastries',

# extra codes from 0708
'Doughnuts, sweet rolls, pastries',
'Cookies, brownies, sweet crackers',
'Cakes and pies'
]

process_foods(category, wweia_desc_filters, num_first_words=2)

### 1.7. Vegetables & Potatoes (not final, additional filtering needed)

In [ ]:
# Load the WWEIA_codes_0708 dataset
wweia_codes_file = "../processed_data/WWEIA_codes_0708.csv"
wweia_codes = pd.read_csv(wweia_codes_file)

# Filter rows where category_desc is "MIXED DISHES"
mixed_dishes = wweia_codes[wweia_codes["group_desc"] == "FRUITS AND VEGETABLES"]

# Extract the WWEIA_desc column as a list
mixed_dishes_list = mixed_dishes["WWEIA_desc"].tolist()

print("List of WWEIA_desc for FRUITS AND VEGETABLES:")
display(mixed_dishes_list)

In [ ]:
category = "vegetables_potatoes"
wweia_desc_filters = [
'Tomatoes',
'Carrots',
'Other red and orange vegetables',
'Broccoli',
'Spinach',
'Lettuce and lettuce salads',
'Other dark green vegetables',
'String beans',
'Cabbage',
'Onions',
'Corn',
'Other starchy vegetables',
'Other vegetables and combinations',
'Fried vegetables',
'Coleslaw, non-lettuce salads',
'Vegetables on a sandwich',
'White potatoes, baked or boiled',
'French fries and other fried white potatoes',
'Mashed potatoes and white potato mixtures',

# extra codes from 0708
'Red and orange vegetables',
'Dark green vegetables',
'Lettuce and lettuce salads',
'Starchy vegetables, excludes white potatoes',
'Other vegetables',
'Vegetable mixed dishes',
'White potatoes',
'White potatoes, fried',
'White potato mixed dishes'
]

process_foods(category, wweia_desc_filters, num_first_words=2)

## Filtering step 2: Not applicable to all food categories:

* This second round of filtering is applicable only to these 3 food categories: meat & seafood, plant protein, vegetables & potatoes. The main goal is to make sure we filter out food items that are not suitable for human consumption (such as raw foods, or foods that have not been processed or cooked sufficiently) and hence, should not be included in a food recommender system. 

* The other 4 categories: mixed dishes, processed meat, baked desserts, breads - need not go through this step since food items in these categories are guaranteed to be suitable for human consumption. 

* The main method of this step is to utilize LLM agents powered by GPT 3.5 Turbo to help us quickly identify foods that are potentially non-recommendable. We can do this simply by prompting the agents with criteria we want to exclude from our final foods list, specifically: 

    - Raw foods
    - Foods that do not include a specific cooking method in their description

In [ ]:
logging.basicConfig(level=logging.ERROR, format='%(asctime)s - %(levelname)s - %(message)s')

# Load environment variables
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# LLM configuration
llm_config = {
    "model": "gpt-3.5-turbo",
    "api_key": api_key
}

# Define the food recommendation agent
agent = ConversableAgent(
    name="food_recommendation_agent",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

# Dynamically create a recommendable criteria based on the dataset
def create_recommendable_criteria(instructions):
    return f"""
    The food description must represent a dish that is suitable for recommendation to users. 
    {instructions}
    
    Recommendable foods are those that can be part of a recipe or a full dish.
    """

# Determine if a food is recommendable
def is_food_recommendable(food_desc, recommendable_criteria, retries=3):
    prompt = f"""
    You are a food recommendation agent. Your task is to judge whether a food description should be recommended to users or not.
    Follow the criteria below:
    {recommendable_criteria}

    Food Description: "{food_desc}"
    Should this food be recommended? Answer "Yes" or "No" with a brief explanation.
    """
    
    for attempt in range(retries):
        try:
            response = agent.generate_reply(
                messages=[{"content": prompt, "role": "user"}]
            )
            answer = response.lower().strip()
            
            if "yes" in answer:
                return "recommendable", answer
            elif "no" in answer:
                return "non-recommendable", answer
            else:
                return "non-recommendable", "Unclear response: " + answer

        except Exception as e:
            # Log the error
            logging.debug(f"Attempt {attempt + 1} failed for '{food_desc}' with error: {e}")
            time.sleep(2 * (attempt + 1))

    return "non-recommendable", "Error during evaluation after retries"

def process_food_file(category, instructions):
    # Create recommendable criteria based on the instructions
    recommendable_criteria = create_recommendable_criteria(instructions)
    
    # Read the input file
    file_path = f'../processed_data/reduced_{category}.csv'
    foods_df = pd.read_csv(file_path)

    # Determine if each food is recommendable
    food_results = []
    for idx, row in foods_df.iterrows():
        food_desc = row['food_desc']
        recommendable_flag, reasoning = is_food_recommendable(food_desc, recommendable_criteria)
        
        food_results.append({
            "food_id": row['food_id'],
            "food_desc": food_desc,
            "WWEIA_desc": row['WWEIA_desc'],
            "ingredient_desc": row.get('ingredient_desc', ''),
            "recommendable_flag": recommendable_flag,
            "reasoning": reasoning
        })
        time.sleep(2)

    food_results_df = pd.DataFrame(food_results)
    
    # Save initial output with recommendations
    intermediate_output_path = f'../processed_data/reduced_{category}_with_recommendations.csv'
    food_results_df.to_csv(intermediate_output_path, index=False)
    print(f"Intermediate results saved to {intermediate_output_path}")

    # Filter rows where recommendable_flag is "recommendable"
    recommendable_df = food_results_df[food_results_df['recommendable_flag'] == "recommendable"]

    # Keep only necessary columns
    recommendable_df = recommendable_df[["food_id", "food_desc", "WWEIA_desc", "ingredient_desc"]]

    # Load the food_tagging data to get nutrition tags
    food_tagging_df = pd.read_csv('../processed_data/food_tagging.csv')

    # Columns to join from food_tagging.csv
    food_tagging_columns = [
        "calorie", "protein", "carb", "sugar", "fiber", "saturated_fat", "cholesterol",
        "sodium", "calcium", "phosphorus", "potassium", "iron", "folic_acid", "vitamin_c",
        "vitamin_d", "vitamin_b12", "low_calorie", "high_calorie", "low_protein", "high_protein",
        "low_carb", "high_carb", "low_sugar", "high_sugar", "low_fiber", "high_fiber",
        "low_saturated_fat", "high_saturated_fat", "low_cholesterol", "high_cholesterol",
        "low_sodium", "high_sodium", "low_calcium", "high_calcium", "low_phosphorus",
        "high_phosphorus", "low_potassium", "high_potassium", "low_iron", "high_iron",
        "low_folic_acid", "high_folic_acid", "low_vitamin_c", "high_vitamin_c", "low_vitamin_d",
        "high_vitamin_d", "low_vitamin_b12", "high_vitamin_b12"
    ]

    # Left join on food_id
    final_df = recommendable_df.merge(food_tagging_df[["food_id"] + food_tagging_columns], on="food_id", how="left")

    print("------------------------------\nList of recommendable foods: ")
    print(f"Number of rows: {len(final_df)}")
    print(f"Number of unique food descriptions: {final_df['food_desc'].nunique()}")
    print(f"Number of unique food id's: {final_df['food_id'].nunique()}")

    output_path = f'../processed_data/reduced_{category}.csv'
    final_df.to_csv(output_path, index=False)
    print(f"------------------------------\nFinal results saved to {output_path}")


In [ ]:
category = "meat_seafood"
instructions = """
Avoid foods that: 
1. Are raw
2. Do not include a specific cooking method in their description
"""

process_food_file(category, instructions)

In [ ]:
category = "plant_protein"
instructions = """
Avoid foods that: 
1. Are raw or unprocessed
2. Do not include a specific cooking method in their description
"""

process_food_file(category, instructions)

In [ ]:
category = "vegetables_potatoes"
instructions = """
Avoid foods that: 
1. Are raw vegetables/potatoes or uncooked vegetables/potatoes
2. Do not include a specific cooking method in their description
"""

process_food_file(category, instructions)